# D23 한국은행 ECOS API 검증

## 목적
한국은행 경제통계시스템(ECOS) Open API를 이용하여
국내 거시경제 지표를 실제로 조회할 수 있는지 검증한다.

### 테스트 지표
- 지표: 원/미국달러 환율
- 기준: 매매기준율
- 통계표 코드: 731Y001
- 항목 코드: 0000001
- 주기: 일별(D)

### 검증 항목
1. ECOS API 연결 가능 여부
2. 최근 12개월 원/달러 환율 조회 가능 여부
3. 날짜·값·단위 확인
4. 결측값 및 조회기간 확인
5. 실제 서비스 활용 시 필요한 통계 코드 기록

> 원/미국달러 매매기준율과 외환시장 종가는 서로 다른 개념이므로 혼동하지 않는다.

In [1]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)

In [2]:
ECOS_KEY = getpass("ECOS API 인증키 입력: ")

STAT_CODE = "731Y001"
ITEM_CODE = "0000001"
CYCLE = "D"

ECOS API 인증키 입력:  ········


In [3]:
def get_ecos_data(
    api_key,
    stat_code,
    cycle,
    start_date,
    end_date,
    item_code,
    start_row=1,
    end_row=1000
):
    """
    한국은행 ECOS StatisticSearch API 호출
    """

    url = (
        f"https://ecos.bok.or.kr/api/StatisticSearch/"
        f"{api_key}/json/kr/"
        f"{start_row}/{end_row}/"
        f"{stat_code}/{cycle}/"
        f"{start_date}/{end_date}/"
        f"{item_code}"
    )

    try:
        response = requests.get(
            url,
            timeout=60
        )

        print("HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print("API 호출 실패")
            print(response.text[:1000])
            return pd.DataFrame()

        data = response.json()

        if "StatisticSearch" not in data:
            print("정상적인 StatisticSearch 응답이 아닙니다.")
            print(data)
            return pd.DataFrame()

        rows = data["StatisticSearch"].get("row", [])

        print(
            "전체 데이터 건수:",
            data["StatisticSearch"].get("list_total_count")
        )

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

## 1. 최근 12개월 원/미국달러 환율 조회

현재 날짜를 기준으로 최근 12개월의 일별 매매기준율을 조회한다.

In [4]:
today = pd.Timestamp.today().normalize()

start_date = (
    today - pd.DateOffset(years=1)
).strftime("%Y%m%d")

end_date = today.strftime("%Y%m%d")

df_ecos = get_ecos_data(
    api_key=ECOS_KEY,
    stat_code=STAT_CODE,
    cycle=CYCLE,
    start_date=start_date,
    end_date=end_date,
    item_code=ITEM_CODE
)

print("조회된 행 수:", len(df_ecos))

display(df_ecos.head())

HTTP 상태코드: 200
전체 데이터 건수: 246
조회된 행 수: 246


,STAT_CODE,STAT_NAME,ITEM_CODE1,ITEM_NAME1,ITEM_CODE2,ITEM_NAME2,ITEM_CODE3,ITEM_NAME3,ITEM_CODE4,ITEM_NAME4,UNIT_NAME,WGT,TIME,DATA_VALUE
0,731Y001,3.1.1.1. 주요국 통화의 대원화환율,0000001,원/미국달러(매매기준율),None,None,None,None,None,None,원,None,20251013,1420.5
1,731Y001,3.1.1.1. 주요국 통화의 대원화환율,0000001,원/미국달러(매매기준율),None,None,None,None,None,None,원,None,20251014,1427.7
2,731Y001,3.1.1.1. 주요국 통화의 대원화환율,0000001,원/미국달러(매매기준율),None,None,None,None,None,None,원,None,20251015,1428.4
3,731Y001,3.1.1.1. 주요국 통화의 대원화환율,0000001,원/미국달러(매매기준율),None,None,None,None,None,None,원,None,20251016,1424.4
4,731Y001,3.1.1.1. 주요국 통화의 대원화환율,0000001,원/미국달러(매매기준율),None,None,None,None,None,None,원,None,20251017,1419.1


In [5]:
print(df_ecos.columns.tolist())

['STAT_CODE', 'STAT_NAME', 'ITEM_CODE1', 'ITEM_NAME1', 'ITEM_CODE2', 'ITEM_NAME2', 'ITEM_CODE3', 'ITEM_NAME3', 'ITEM_CODE4', 'ITEM_NAME4', 'UNIT_NAME', 'WGT', 'TIME', 'DATA_VALUE']


In [6]:
wanted_cols = [
    "TIME",
    "DATA_VALUE",
    "STAT_NAME",
    "ITEM_NAME1",
    "UNIT_NAME"
]

existing_cols = [
    col for col in wanted_cols
    if col in df_ecos.columns
]

exchange_rate = df_ecos[
    existing_cols
].copy()

exchange_rate = exchange_rate.rename(columns={
    "TIME": "날짜",
    "DATA_VALUE": "환율",
    "STAT_NAME": "통계명",
    "ITEM_NAME1": "항목명",
    "UNIT_NAME": "단위"
})

exchange_rate["날짜"] = pd.to_datetime(
    exchange_rate["날짜"],
    format="%Y%m%d",
    errors="coerce"
)

exchange_rate["환율"] = pd.to_numeric(
    exchange_rate["환율"],
    errors="coerce"
)

exchange_rate = (
    exchange_rate
    .sort_values("날짜")
    .reset_index(drop=True)
)

display(exchange_rate.head())

,날짜,환율,통계명,항목명,단위
0,2025-10-13,1420.5,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
1,2025-10-14,1427.7,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
2,2025-10-15,1428.4,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
3,2025-10-16,1424.4,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
4,2025-10-17,1419.1,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원


In [7]:
display(exchange_rate.tail(20))

,날짜,환율,통계명,항목명,단위
226,2026-09-10,1337.9,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
227,2026-09-11,1338.2,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
228,2026-09-14,1346.4,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
229,2026-09-15,1345.3,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
230,2026-09-16,1353.3,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
231,2026-09-17,1368.3,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
232,2026-09-18,1380.3,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
233,2026-09-21,1383.8,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
234,2026-09-22,1384.3,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원
235,2026-09-23,1360.0,3.1.1.1. 주요국 통화의 대원화환율,원/미국달러(매매기준율),원


## 2. 데이터 품질 검증

최근 12개월 동안 조회기간, 결측값 및 중복 날짜를 확인한다.

In [8]:
print("===== D23 ECOS 검증 결과 =====")

print("데이터 존재:", not exchange_rate.empty)
print("조회 건수:", len(exchange_rate))

if not exchange_rate.empty:

    print(
        "조회기간:",
        exchange_rate["날짜"].min(),
        "~",
        exchange_rate["날짜"].max()
    )

    print(
        "중복 날짜:",
        exchange_rate["날짜"].duplicated().sum()
    )

    print(
        "환율 결측값:",
        exchange_rate["환율"].isna().sum()
    )

    print(
        "통계명:",
        exchange_rate["통계명"].dropna().unique()
    )

    print(
        "항목명:",
        exchange_rate["항목명"].dropna().unique()
    )

    print(
        "단위:",
        exchange_rate["단위"].dropna().unique()
    )

===== D23 ECOS 검증 결과 =====
데이터 존재: True
조회 건수: 246
조회기간: 2025-10-13 00:00:00 ~ 2026-10-13 00:00:00
중복 날짜: 0
환율 결측값: 0
통계명: <ArrowStringArray>
['3.1.1.1. 주요국 통화의 대원화환율']
Length: 1, dtype: str
항목명: <ArrowStringArray>
['원/미국달러(매매기준율)']
Length: 1, dtype: str
단위: <ArrowStringArray>
['원']
Length: 1, dtype: str
